# Hybrid Semantic CBF + CF - Sistem Rekomendasi Pembimbing Proposal Skripsi

## -- Install Dependencies

In [1]:
!pip install sentence-transformers scikit-learn pandas numpy --quiet

In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.optimize import linear_sum_assignment
import re #buat regex

## -- Configurasi

In [3]:
CONFIG = {
    # --- PATH FILE (ganti sesuai lokasi file kamu) ---
    "path_historis": "data_historis.csv",
    "path_terbaru": "data_test.csv",
    "path_dosen_master": "data_dosen.csv",

    # --- MAPPING KOLOM: dataset historis ---
    "col_historis_nim": "nim",
    "col_historis_nama_mhs": "nama_mahasiswa",
    "col_historis_judul": "judul",
    "col_historis_tahun": "tahun_terbit",
    "col_historis_dospem1": "dospem1",
    "col_historis_dospem2": "dospem2",

    # --- MAPPING KOLOM: dataset angkatan terbaru (test set / ground truth) ---
    "col_terbaru_nim": "nim",
    "col_terbaru_nama_mhs": "nama_mahasiswa",
    "col_terbaru_judul": "judul",
    "col_terbaru_dospem_aktual": "dosen",

    # --- MAPPING KOLOM: dataset dosen master ---
    "col_dosen_id": "id_dosen",
    "col_dosen_nama": "nama_dosen",
    "col_dosen_bidang": "bidang_konsentrasi",
    "col_dosen_kata_kunci": "keyword",
    "col_dosen_status_aktif": "status",

    # --- PARAMETER EKSPERIMEN ---
    "model_name": "paraphrase-multilingual-mpnet-base-v2",
    "top_n_historis": [2, 3, 4, 5, 6, 7, 8, 9, 10],        # N: jumlah judul historis paling mirip yang diambil untuk CF
    "role_weight_dospem1": 1.0,
    "role_weight_dospem2": [0.0],
    "lambda_decay": [0.0],         # decay rate untuk recency_weight
    "tahun_sekarang": 2026,
    "alpha_list": [0.0, 0.1, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.75, 0.8, 0.9, 1.0],  # untuk tuning hybrid CBF + CF
    "beta_list": [0.0, 0.1, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.7, 0.75, 0.8, 0.9, 1.0], # untuk tuning hybrid TF-IDF + SBERT
    "k_eval": [1, 3, 5],          # untuk Precision@K / Recall@K
    "primary_metric": "HitRate@1",
}


## -- Load Data, Filter Status Dosen

In [4]:
def load_table(path):
  if path.endswith(".csv"):
    return pd.read_csv(path)
  return pd.read_excel(path)

df_historis = load_table(CONFIG["path_historis"])
df_terbaru = load_table(CONFIG["path_terbaru"])
df_dosen = load_table(CONFIG["path_dosen_master"])

print("Historis:", df_historis.shape)
print("Terbaru:", df_terbaru.shape)
print("Dosen:", df_dosen.shape)
df_historis.head()

Historis: (482, 6)
Terbaru: (124, 4)
Dosen: (38, 5)


,nim,nama_mahasiswa,judul,tahun_terbit,dospem1,dospem2
0,M0519051,Irfan Rafi Rizaldi,Pengenalan Emosi dan Estimasi Usia Menggunakan...,2026,SARI WIDYA SIHWI,ERY PERMANA YUDHA
1,M0519052,Joy Kristian Eldo,Segmentasi Citra Inti Leukosit Menggunakan Met...,2026,ESTI SURYANI,UMI SALAMAH
2,M0519060,Muhammad Dwi Arfian,Image Captioning pada Citra Halftone dengan Fi...,2026,DEWI WISNU WARDANI,NaN
3,M0519086,Zidane Hidayat,"Analisa Komparatif Pengaruh Augmentasi GPT-2, ...",2026,HASAN DWI CAHYONO,FAJAR MUSLIM
4,M0520001,Abdurrahman Zufar,Augmentasi Masking pada Mel-Spectrogram dan MF...,2026,SARI WIDYA SIHWI,HERDITO IBNU DEWANGKORO


In [5]:
def to_bool_status(val):
  if isinstance(val, bool):
    return val
  if isinstance(val, (int, float)):
    return bool(val)
  val = str(val).strip().lower()
  return val in ('true', '1', 'aktif', 'ya', 'yes')

df_dosen["_status_aktif"] = df_dosen[CONFIG["col_dosen_status_aktif"]].apply(to_bool_status)

df_dosen_aktif = df_dosen[df_dosen["_status_aktif"] == True].reset_index(drop=True)
print(f"Total dosen master: {len(df_dosen)}, dosen aktif (lolos filter): {len(df_dosen_aktif)}")
df_dosen_aktif[[CONFIG["col_dosen_id"], CONFIG["col_dosen_nama"]]]

Total dosen master: 38, dosen aktif (lolos filter): 19


,id_dosen,nama_dosen
0,Dsn_2,AFRIZAL DOEWES
1,Dsn_3,AKHMAD SYAIFUDDIN
2,Dsn_7,ARIF ROHMADI
3,Dsn_9,BAMBANG HARJITO
4,Dsn_11,BRILYAN HENDRASURYAWAN
5,Dsn_13,DEWI WISNU WARDANI
6,Dsn_16,ERY PERMANA YUDHA
7,Dsn_17,ESTI SURYANI
8,Dsn_18,FAJAR MUSLIM
9,Dsn_20,HASAN DWI CAHYONO


## -- Semantic Embedding (SBERT)

Encode judul mahasiswa (historis & terbaru) dan profil dosen aktif (Bidang_Konsentrasi + Kata_Kunci)
ke dalam embedding space yang sama.

*   emb_historis : Judul TA lama
*   emb_test : Judul mahasiswa yang akan direkomendasikan
*   emb_dosen : Gabungan bidang konsentrasi + keyword tiap dosen


In [6]:
model = SentenceTransformer(CONFIG["model_name"])

#embedding judul historis
judul_historis = df_historis[CONFIG["col_historis_judul"]].fillna("").tolist()
emb_historis = model.encode(judul_historis, show_progress_bar=True, normalize_embeddings=True)

#embedding judul mahasiswa test
judul_test = df_terbaru[CONFIG["col_terbaru_judul"]].fillna("").tolist()
emb_test = model.encode(judul_test, show_progress_bar=True, normalize_embeddings=True)

#embedding profil dosen aktif (gabungan bidang_konsentrasi + kata_kunci)
profil_dosen = (
    df_dosen_aktif[CONFIG["col_dosen_bidang"]].fillna("") + ". " +
    df_dosen_aktif[CONFIG["col_dosen_kata_kunci"]].fillna("")
).tolist()
emb_dosen = model.encode(profil_dosen, show_progress_bar=True, normalize_embeddings=True)

print(f"Embedding judul historis: {emb_historis.shape}")
print(f"Embedding judul mahasiswa test: {emb_test.shape}")
print(f"Embedding profil dosen aktif: {emb_dosen.shape}")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding judul historis: (482, 768)
Embedding judul mahasiswa test: (124, 768)
Embedding profil dosen aktif: (19, 768)


## --TF-IDF Embedding

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

corpus_cbf = judul_test + profil_dosen

tfidf_cbf_vec = TfidfVectorizer()
tfidf_cbf_matrix = tfidf_cbf_vec.fit_transform(corpus_cbf)

n_test = len(judul_test)
tfidf_mhs_cbf = normalize(tfidf_cbf_matrix[:n_test])
tfidf_dosen_cbf = normalize(tfidf_cbf_matrix[n_test:])

tfidf_cbf_matrix_scores = cosine_similarity(tfidf_mhs_cbf, tfidf_dosen_cbf)

print(f"TF-IDF CBF matrix shape: {tfidf_cbf_matrix_scores.shape}")

TF-IDF CBF matrix shape: (124, 19)


## -- CBF Score

Cosine similarity antara embedding judul mahasiswa (test set) vs embedding profil dosen aktif.
Hasil: matriks berukuran (jumlah mahasiswa test × jumlah dosen aktif).

In [9]:
sbert_cbf_matrix = cosine_similarity(emb_test, emb_dosen)

beta_check =0.5
cbf_matrix_check = beta_check * tfidf_cbf_matrix_scores + (1 - beta_check) * sbert_cbf_matrix

cbf_df = pd.DataFrame(
    cbf_matrix_check,
    index=df_terbaru[CONFIG["col_terbaru_nim"]],
    columns=df_dosen_aktif[CONFIG["col_dosen_id"]]
)

print(f"CBF marix (beta={beta_check}) shape: {cbf_matrix_check.shape}")
cbf_df.head(10)

CBF marix (beta=0.5) shape: (124, 19)


id_dosen,Dsn_2,Dsn_3,Dsn_7,Dsn_9,Dsn_11,Dsn_13,Dsn_16,Dsn_17,Dsn_18,Dsn_20,Dsn_21,Dsn_22,Dsn_23,Dsn_28,Dsn_29,Dsn_32,Dsn_33,Dsn_35,Dsn_37
nim,,,,,,,,,,,,,,,,,,,
L0123013,0.132067,0.127249,0.157094,0.050160,0.064517,0.103980,0.157093,0.148863,0.120586,0.140989,0.134723,0.152599,0.109761,0.101637,0.160810,0.127913,0.150397,0.125684,0.159840
L0123110,0.185055,0.116870,0.256071,0.077596,0.076643,0.188224,0.210738,0.112590,0.427437,0.155250,0.218730,0.219085,0.210364,0.109944,0.197217,0.135597,0.195945,0.187954,0.191005
L0123040,0.209227,0.111063,0.142753,0.044759,0.095418,0.118823,0.225471,0.076623,0.218675,0.122942,0.133309,0.112102,0.180980,0.101777,0.122256,0.095893,0.118439,0.103906,0.123745
L0123051,0.175217,0.148604,0.175754,0.038888,0.088188,0.134023,0.136516,0.105838,0.217157,0.132132,0.140126,0.113066,0.126389,0.139541,0.210574,0.147363,0.102807,0.145151,0.117925
L0123088,0.044894,0.015147,0.062303,-0.014772,-0.012384,0.052185,0.134025,0.143859,0.054684,0.078684,0.086724,0.165528,0.117529,0.064132,0.080296,0.034552,0.078014,0.095908,0.037565
L0123020,0.183957,0.060661,0.096238,0.084854,0.066663,0.126909,0.230610,0.074893,0.225170,0.089769,0.105135,0.070984,0.167700,0.111683,0.092384,0.041082,0.087597,0.067680,0.085220
L0123023,0.258734,0.147005,0.232940,0.158900,0.067439,0.156035,0.211584,0.141295,0.250090,0.167267,0.183435,0.189484,0.255713,0.188639,0.194359,0.132910,0.150702,0.193413,0.164825
L0123018,0.246206,0.169886,0.174475,0.072577,0.109875,0.187876,0.259812,0.094319,0.259750,0.132079,0.149216,0.145757,0.231239,0.113196,0.149404,0.167381,0.140733,0.126103,0.151374
L0123096,0.157861,0.139678,0.193435,0.078477,0.073857,0.074679,0.137364,0.084962,0.157824,0.099484,0.145390,0.118894,0.124310,0.111482,0.237175,0.080469,0.083007,0.129287,0.092318


## -- CF Score

Untuk setiap mahasiswa test:
1. Hitung similarity ke semua judul historis.
2. Ambil Top-N paling mirip.
3. Agregasi per dosen dengan `recency_weight` dan `role_weight`.
4. Normalisasi (min-max) agar sebanding dengan CBF score.


In [10]:
sim_terbaru_vs_historis = cosine_similarity(emb_test, emb_historis)  # (n_mhs_test, n_judul_historis)

dosen_id_list = df_dosen_aktif[CONFIG["col_dosen_id"]].tolist()
dosen_id_to_idx = {d: i for i, d in enumerate(dosen_id_list)}
nama_to_id = dict(zip(df_dosen_aktif[CONFIG["col_dosen_nama"]], df_dosen_aktif[CONFIG["col_dosen_id"]]))

tahun_historis = df_historis[CONFIG["col_historis_tahun"]].values
dospem1_name = df_historis[CONFIG["col_historis_dospem1"]].values
dospem2_name = df_historis[CONFIG["col_historis_dospem2"]].values
tahun_sekarang = CONFIG["tahun_sekarang"]
w1 = CONFIG["role_weight_dospem1"]

n_mhs_test = len(df_terbaru)
n_dosen_aktif = len(dosen_id_list)

def minmax_normalize_rows(mat):
    mn = mat.min(axis=1, keepdims=True)
    mx = mat.max(axis=1, keepdims=True)
    denom = np.where((mx - mn) == 0, 1, mx - mn)
    return (mat - mn) / denom

def compute_cf_matrix(N, lam, w2):
  cf_raw = np.zeros((n_mhs_test, n_dosen_aktif))

  for i in range(n_mhs_test):
    sims = sim_terbaru_vs_historis[i]
    top_idx = np.argsort(sims)[::-1][:N]
    for j in top_idx:
      sim_val = sims[j]
      recency_weight = np.exp(-lam * (tahun_sekarang - tahun_historis[j]))

      d1 = dospem1_name[j]
      d2 = dospem2_name[j]

      if d1 in nama_to_id:
        idx1 = dosen_id_to_idx[nama_to_id[d1]]
        cf_raw[i, idx1] += w1 * sim_val * recency_weight * w1

      if d2 in nama_to_id:
        idx2 = dosen_id_to_idx[nama_to_id[d2]]
        cf_raw[i, idx2] += w1 * sim_val * recency_weight * w2
  return minmax_normalize_rows(cf_raw)

cf_matrix_check = compute_cf_matrix(N=10, lam=0.3, w2=0.6)
print("Shape CF matrix (sanity check):", cf_matrix_check.shape)

Shape CF matrix (sanity check): (124, 19)


**Catatan cold-start:** dosen yang tidak punya histori sama sekali otomatis bernilai 0 di `cf_matrix_raw`
sebelum normalisasi, sehingga kontribusinya ke skor hybrid nanti murni datang dari CBF (sesuai desain fallback
di Research Method).

## -- Hybrid Scoring & Tuning Alpha

`Final_score = alpha * CBF_score + (1-alpha) * CF_score`

Dievaluasi pakai Precision@K dan Recall@K terhadap `Dospem_Aktual` (ground truth) untuk tiap nilai alpha,
termasuk alpha=0 (CF-only) dan alpha=1 (CBF-only) sebagai skenario pembanding.

In [20]:
import itertools

def get_topk_dosen_ids(score_matrix, dosen_ids, k):
    topk_results = []
    for row in score_matrix:
        idx_sorted = np.argsort(row)[::-1][:k]
        topk_results.append([dosen_ids[i] for i in idx_sorted])
    return topk_results


def hitrate_at_k(score_matrix, dosen_ids, ground_truth_ids, k):
    topk = get_topk_dosen_ids(score_matrix, dosen_ids, k)
    hits = [1 if gt in pred else 0 for gt, pred in zip(ground_truth_ids, topk)]
    return np.mean(hits)


In [12]:
# Mapping ground truth Dospem_Aktual (nama bersih) -> Dosen_ID
gt_dosen_ids = []
valid_rows = []  # index mahasiswa test yang ground truth-nya berhasil dimapping ke dosen aktif
for i, nama in enumerate(df_terbaru[CONFIG["col_terbaru_dospem_aktual"]]):
    if nama in nama_to_id:
        gt_dosen_ids.append(nama_to_id[nama])
        valid_rows.append(i)

print(f"Mahasiswa test yang valid untuk evaluasi: {len(valid_rows)} dari {n_mhs_test}")

cbf_eval = cbf_matrix_check[valid_rows]

Mahasiswa test yang valid untuk evaluasi: 124 dari 124


In [13]:
grid_results = []

param_combinations = list(itertools.product(
    CONFIG["top_n_historis"],
    CONFIG["role_weight_dospem2"],
    CONFIG["lambda_decay"],
    CONFIG["beta_list"]
))
print(f"Total kombinasi: {len(param_combinations)} x {len(CONFIG['alpha_list'])} alpha = {len(param_combinations) * len(CONFIG['alpha_list'])} eksperimen")

for N, w2, lam, beta in param_combinations:
  cbf_matrix = beta * tfidf_cbf_matrix_scores + (1 - beta) * sbert_cbf_matrix

  cf_mat = compute_cf_matrix(N=N, lam=lam, w2=w2)

  cbf_eval = cbf_matrix[valid_rows]
  cf_eval = cf_mat[valid_rows]

  for alpha in CONFIG["alpha_list"]:
    final_mat = alpha * cbf_eval + (1-alpha) * cf_eval
    row_result = {"N": N, "w2": w2, "lam": lam, "beta": beta, "alpha": alpha}
    for k in CONFIG["k_eval"]:
      row_result[f"HitRate@{k}"] = hitrate_at_k(final_mat, dosen_id_list, gt_dosen_ids, k)
    grid_results.append(row_result)

grid_df = pd.DataFrame(grid_results)
grid_df.sort_values(CONFIG["primary_metric"], ascending=False).head(10)

Total kombinasi: 117 x 13 alpha = 1521 eksperimen


,N,w2,lam,beta,alpha,HitRate@1,HitRate@3,HitRate@5
518,5,0.0,0.0,0.00,0.90,0.193548,0.362903,0.475806
869,7,0.0,0.0,0.10,0.90,0.185484,0.346774,0.467742
856,7,0.0,0.0,0.00,0.90,0.185484,0.362903,0.467742
180,3,0.0,0.0,0.00,0.90,0.185484,0.338710,0.443548
349,4,0.0,0.0,0.00,0.90,0.185484,0.338710,0.459677
256,3,0.0,0.0,0.50,0.75,0.177419,0.306452,0.403226
687,6,0.0,0.0,0.00,0.90,0.177419,0.370968,0.467742
230,3,0.0,0.0,0.30,0.75,0.177419,0.306452,0.427419
475,4,0.0,0.0,0.80,0.60,0.177419,0.290323,0.435484
217,3,0.0,0.0,0.25,0.75,0.177419,0.306452,0.427419


In [28]:
best_row = (
    grid_df
    .sort_values(
        by=["HitRate@5"],
        ascending=False
    )
    .iloc[0]
)
#best_row = grid_df.loc[grid_df[CONFIG["primary_metric"]].idxmax()]
best_N = int(best_row["N"])
best_w2 = best_row["w2"]
best_lambda = best_row["lam"]
best_alpha = best_row["alpha"]
best_beta = best_row["beta"]

print("=== Kombinasi parameter TERBAIK ===")
print(best_row)

=== Kombinasi parameter TERBAIK ===
N            6.000000
w2           0.000000
lam          0.000000
beta         0.400000
alpha        0.900000
HitRate@1    0.161290
HitRate@3    0.346774
HitRate@5    0.491935
Name: 752, dtype: float64


In [27]:
# === Tempel cell ini SETELAH hybrid_matrix, gt_dosen_ids, valid_rows sudah dihitung ===
# (letakkan setelah cell 22 "Kombinasi parameter TERBAIK", pastikan variabel
#  hybrid_matrix dihitung ulang pakai best_N/best_alpha/best_beta jika belum ada)

id_to_name = dict(zip(df_dosen_aktif[CONFIG["col_dosen_id"]], df_dosen_aktif[CONFIG["col_dosen_nama"]]))

def get_topk_and_rank(score_matrix, dosen_ids, gt_ids, valid_idx, k=5):
    out = []
    for pos, (i, gt) in enumerate(zip(valid_idx, gt_ids)):
        row = score_matrix[pos] # Use pos because score_matrix is already filtered by valid_idx
        order = np.argsort(row)[::-1]
        sorted_dosen = [dosen_ids[j] for j in order]
        rank = sorted_dosen.index(gt) + 1
        topk = sorted_dosen[:k]
        out.append((i, topk, gt, rank))
    return out

results = get_topk_and_rank(final_hybrid, dosen_id_list, gt_dosen_ids, valid_rows, k=5)

# ambil 1 contoh rank=1, 1 contoh rank 2-3, 1 contoh rank>5 (kalau ada)
picked = {"rank1": None, "rank_mid": None, "rank_jauh": None}
for i, topk, gt, rank in results:
    if rank == 1 and picked["rank1"] is None:
        picked["rank1"] = (i, topk, gt, rank)
    elif 2 <= rank <= 3 and picked["rank_mid"] is None:
        picked["rank_mid"] = (i, topk, gt, rank)
    elif rank > 5 and picked["rank_jauh"] is None:
        picked["rank_jauh"] = (i, topk, gt, rank)

for label, val in picked.items():
    if val is None:
        continue
    i, topk, gt, rank = val
    nim = df_terbaru.iloc[i][CONFIG["col_terbaru_nim"]]
    judul = df_terbaru.iloc[i][CONFIG["col_terbaru_judul"]]
    print(f"\n[{label}] NIM={nim}")
    print("Judul:", judul)
    print("Top-5 rekomendasi:", [id_to_name.get(d, d) for d in topk])
    print("Dosen aktual (ground truth):", id_to_name.get(gt, gt))
    print("Rank ground truth:", rank, " RR =", round(1/rank, 3))


[rank1] NIM=L0123056
Judul: PENERAPAN METODE BLACK-BOX PENETRATION TESTING UNTUK SECURITY ASSESSMENT PADA APLIKASI WEBFATISDA.UNS.AC.IDD
Top-5 rekomendasi: ['BAMBANG HARJITO', 'RINI ANGGRAININGSIH', 'WIHARTO', 'RISTU SAPTONO', 'M FAHMY NADHIF']
Dosen aktual (ground truth): BAMBANG HARJITO
Rank ground truth: 1  RR = 1.0

[rank_mid] NIM=L0123022
Judul: Analisis Sentimen dan Deteksi Mismatch Rating pada Ulasan Produk Shopee Menggunakan Naive Bayes dan TF-IDF dengan Penanganan Imbalanced Data Menggunakan SMOTE
Top-5 rekomendasi: ['RISTU SAPTONO', 'ARIF ROHMADI', 'FAJAR MUSLIM', 'AFRIZAL DOEWES', 'RINI ANGGRAININGSIH']
Dosen aktual (ground truth): ARIF ROHMADI
Rank ground truth: 2  RR = 0.5

[rank_jauh] NIM=L0123013
Judul: DETEKSI SARKASME PADA TEKS ULASAN MENGGUNAKAN ARSITEKTUR INDOBERT BERBASIS CONTEXTUAL COLLOCATION AUGMENTATION
Top-5 rekomendasi: ['RISTU SAPTONO', 'WIRANTO', 'ARIF ROHMADI', 'ERY PERMANA YUDHA', 'WIHARTO']
Dosen aktual (ground truth): AFRIZAL DOEWES
Rank ground truth: 1

## -- Baseline TF-IDF

untuk mereplikasi paper rujukan

In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer

corpus_tfidf = judul_test + profil_dosen
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(corpus_tfidf)

n_test = len(judul_test)
tfidf_mhs = tfidf_matrix[:n_test]
tfidf_dosen =  tfidf_matrix[n_test:]

baseline_matrix = cosine_similarity(tfidf_mhs, tfidf_dosen)
baseline_eval = baseline_matrix[valid_rows]

baseline_results = {"scenario": "Baseline TF-IDF"}
for k in CONFIG["k_eval"]:
  baseline_results[f"HitRate@{k}"] = hitrate_at_k(baseline_eval, dosen_id_list, gt_dosen_ids, k)

print(baseline_results)

{'scenario': 'Baseline TF-IDF', 'HitRate@1': np.float64(0.10483870967741936), 'HitRate@3': np.float64(0.22580645161290322), 'HitRate@5': np.float64(0.4032258064516129)}


## -- Baseline vs CBF-only bs CF-only vs Hybrid

In [16]:
def get_rank_of_ground_truth(score_matrix, dosen_ids, ground_truth_ids):
  ranks = []
  for row, gt in zip(score_matrix, ground_truth_ids):
    order = np.argsort(row)[::-1]
    sorted_dosen = [dosen_ids[i] for i in order]
    try:
      rank = sorted_dosen.index(gt) + 1
    except ValueError:
      rank = np.inf
    ranks.append(rank)
  return np.array(ranks)

def mrr(ranks):
  reciprocal = np.where(np.isinf(ranks), 0.0, 1.0 / ranks)
  return reciprocal.mean()

In [22]:
def evaluate_full(score_matrix, dosen_ids, ground_truth_ids, k_list):
    ranks = get_rank_of_ground_truth(score_matrix, dosen_ids, ground_truth_ids)
    mean_reciprocal_rank = mrr(ranks)

    results = {"MRR": mean_reciprocal_rank}
    for k in k_list:
        results[f"Recall@{k}"] = hitrate_at_k(score_matrix, dosen_ids, ground_truth_ids, k)

    return results, ranks

k_list = CONFIG["k_eval"]

eval_results ={}
ranks_per_scenario = {}

# Recalculate cbf_eval_best and cf_eval_best using optimal parameters
cbf_matrix_best = best_beta * tfidf_cbf_matrix_scores + (1 - best_beta) * sbert_cbf_matrix
cbf_eval_best = cbf_matrix_best[valid_rows]

cf_mat_best = compute_cf_matrix(N=best_N, lam=best_lambda, w2=best_w2)
cf_eval_best = cf_mat_best[valid_rows]

# Baseline TF-IDF
res, ranks = evaluate_full(baseline_eval, dosen_id_list, gt_dosen_ids, k_list)
eval_results["Baseline TF-IDF"] = res
ranks_per_scenario["Baseline TF-IDF"] = ranks

# CBF-only
res, ranks = evaluate_full(cbf_eval_best, dosen_id_list, gt_dosen_ids, k_list)
eval_results["CBF-only"] = res
ranks_per_scenario["CBF-only"] = ranks

# CF-only
res, ranks = evaluate_full(cf_eval_best, dosen_id_list, gt_dosen_ids, k_list)
eval_results["CF-only"] = res
ranks_per_scenario["CF-only"] = ranks

# Hybrid
final_hybrid = best_alpha * cbf_eval_best + (1-best_alpha) * cf_eval_best
res, ranks = evaluate_full(final_hybrid, dosen_id_list, gt_dosen_ids, k_list)
eval_results["Hybrid"] = res
ranks_per_scenario["Hybrid"] = ranks

In [23]:
specific_scenarios = [
    "Baseline TF-IDF",  # TF-IDF + Cosine similarity (tanpa semantic embedding) + CBF
    "CBF-only",         # TF-IDF + Semantic Similarity + CBF (fusion TF-IDF dan SBERT untuk CBF)
    "CF-only",          # CBF-only (Collaborative Filtering only)
    "Hybrid"            # TF-IDF + Semantic Similarity + CBF + CF (hybrid)
]

comparison_data = []
for scenario_name in specific_scenarios:
    if scenario_name in eval_results:
        metrics = eval_results[scenario_name]
        row = {"Skenario": scenario_name}
        row.update({
            "MRR": metrics["MRR"],
            "HitRate@1": metrics["Recall@1"], # Recall@K is equivalent to HitRate@K
            "HitRate@3": metrics["Recall@3"],
            "HitRate@5": metrics["Recall@5"]
        })
        comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df)

,Skenario,MRR,HitRate@1,HitRate@3,HitRate@5
0,Baseline TF-IDF,0.245640,0.104839,0.225806,0.403226
1,CBF-only,0.295831,0.129032,0.314516,0.435484
2,CF-only,0.276784,0.120968,0.306452,0.467742
3,Hybrid,0.325389,0.161290,0.346774,0.491935
